## RAG Agent Notebook

Interactive step-by-step testing of `rag_agent.py` components.


In [1]:
# Limit threads to avoid potential FAISS/OMP issues
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import huggingface_hub
if not hasattr(huggingface_hub, "cached_download"):
    huggingface_hub.cached_download = huggingface_hub.hf_hub_download

import glob
import yaml
import json         # ← add this
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from smolagents import CodeAgent, Tool, InferenceClientModel

/Users/shanzhonghan/Desktop/LLM/project/juris/backend/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load config
with open("config.yaml", encoding="utf8") as f:
    cfg = yaml.safe_load(f)
cfg


{'chunk_size': 800,
 'chunk_overlap': 200,
 'embed_model': 'all-MiniLM-L6-v2',
 'gen_model': 'google/flan-t5-small',
 'cross_encoder_model': 'cross-encoder/ms-marco-MiniLM-L-12-v2',
 'initial_top_k': 10,
 'rerank_top_k': 3,
 'sitemap_root': 'https://www.retsinformation.dk/eli/sitemap.xml',
 'out_dir': 'laws_json',
 'index_path': 'laws_mini.faiss',
 'meta_path': 'laws_mini_meta.json',
 'host': '0.0.0.0',
 'port': 8000,
 'log_level': 'INFO',
 'log_format': '%(asctime)s %(levelname)s %(name)s: %(message)s'}

In [3]:
# Load vector index & metadata
INDEX = faiss.read_index(cfg["index_path"])
with open(cfg["meta_path"], encoding="utf8") as f:
    META = json.load(f)
len(META)


27076

In [4]:
# Load full law JSONs
LAW_DOCS = {}
for path in glob.glob(os.path.join(cfg["out_dir"], "*.json")):
    with open(path, encoding="utf8") as f:
        doc = json.load(f)
    LAW_DOCS[doc["id"]] = doc
len(LAW_DOCS)


494

In [5]:
# Define load_chunk_text
def load_chunk_text(hit: dict) -> str:
    law = LAW_DOCS.get(hit["law_id"], {})
    for chap in law.get("structured_text", []):
        for para in chap.get("paragraphs", []):
            if para["paragraph"] == hit["paragraph"]:
                for sec in para["sections"]:
                    if sec["section"] == hit["section"]:
                        text = sec.get("text", "")
                        start = hit.get("char_offset", 0)
                        return text[start : start + cfg["chunk_size"]]
    return ""

# Example:
# load_chunk_text(META[0])


# Example query
load_chunk_text(META[0])

'Personer, der udfører asbestarbejde som autorisationskrævende nedrivning af asbestholdigt materiale, jf. § 49 e, stk. 1, i lov om arbejdsmiljø og bekendtgørelse om asbest i arbejdsmiljøet, skal have gennemgået en uddannelse og være i besiddelse af et uddannelsesbevis. Det fremgår af bilag 6, afsnit 1, hvilke kvalifikationer man skal være i besiddelse af, før der kan udstedes et uddannelsesbevis.'

In [6]:
# Load models
EMBEDDER      = SentenceTransformer(cfg["embed_model"])
CROSS_ENCODER = CrossEncoder(cfg["cross_encoder_model"])
TOKENIZER     = AutoTokenizer.from_pretrained(cfg["gen_model"])
GEN_MODEL     = AutoModelForSeq2SeqLM.from_pretrained(cfg["gen_model"])
GEN_PIPE      = pipeline("text2text-generation", model=GEN_MODEL, tokenizer=TOKENIZER)
EMBEDDER, CROSS_ENCODER, GEN_PIPE


Device set to use cpu


(SentenceTransformer(
   (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
   (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False})
   (2): Normalize()
 ),
 <transformers.pipelines.text2text_generation.Text2TextGenerationPipeline at 0x12864fb90>)

In [7]:
class RAGTool(Tool):
    name        = "rag_tool"
    description = "Retrieve and answer questions about Danish law using RAG."
    inputs = {
        "query": {
            "type":        "string",
            "description": "The user's legal question",
            "required":    True,
        },
    }
    output_type = "object"

    def forward(self, query: str):
        # 永远使用 config 中的默认值
        top_k    = cfg["initial_top_k"]
        rerank_k = cfg["rerank_top_k"]

        # 1) Retrieve via FAISS + embedder
        vec        = np.array(EMBEDDER.encode([query]), dtype="float32")
        dists, idxs = INDEX.search(vec, top_k)
        hits       = [{**META[i], "distance": float(d)} for d, i in zip(dists[0], idxs[0])]

        # 2) Rerank via CrossEncoder
        texts   = [load_chunk_text(h) for h in hits]
        scores  = CROSS_ENCODER.predict([[query, t] for t in texts])
        top_hits = [
            h for h, _ in sorted(zip(hits, scores),
                                 key=lambda x: x[1], reverse=True)[:rerank_k]
        ]

        # 3) Generate answer with citations in prompt
        context = "\n\n".join(
            f"[Lov {h['law_id']} §{h['paragraph']} stk.{h['section']}] {load_chunk_text(h)}"
            for h in top_hits
        )
        prompt = (
            "You are a Danish legal assistant. Answer concisely using the excerpts below and cite.\n\n"
            f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
        )
        out = GEN_PIPE(
            prompt,
            max_length=512,
            do_sample=False,
            num_beams=4
        )[0]["generated_text"].strip()

        return {"answer": out, "citations": top_hits}
rag_tool = RAGTool()

In [8]:
result = rag_tool.forward("Hvordan skal HR håndtere en medarbejder, der er blevet pålagt elektronisk kontrol som del af et opholdsforbud, så vi sikrer, at både virksomheden og medarbejderen overholder alle relevante krav og procedurer?")
print(result)


Token indices sequence length is longer than the specified maximum sequence length for this model (612 > 512). Running this sequence through the model will result in indexing errors


{'answer': '[Lov 2025_681  47 d. stk.Stk. 3.] Erhvervsstyrelsen skal meddele udbyderen sine forelbige konklusioner om, hvorvidt de afgivne tilsagn opfylder de ml og kriterier og overholder de procedurer, som er omhandlet i nrvrende bestemmelse og i  41, 47 a eller 47 c, og p hvilke betingelser styrelsen kan overveje and gre tilsagnene bindende. [Lov 2025_964  4. stk.Stk.] Vejdirektoratet udarbejder og ajourfrer en handlingsplan med henblik p at flge gennemfrelsen af foranstaltninger, hvad enten disse er prioriteret ud fra bestemmelserne i  5, stk. 4, eller som resultat af de mlrettede trafiksikkerhedsinspektioner, jf.  5, stk. 5.', 'citations': [{'law_id': '2025_681', 'chapter': 'Kapitel 14 a', 'paragraph': '§ 47 d.', 'section': 'Stk. 3.', 'char_offset': 0, 'distance': 0.6101877689361572}, {'law_id': '2025_964', 'chapter': '', 'paragraph': '§ 4.', 'section': 'Stk. 4.', 'char_offset': 0, 'distance': 0.5933927297592163}, {'law_id': '2025_946', 'chapter': '', 'paragraph': '§ 6.', 'section

In [9]:
model     = InferenceClientModel()
rag_agent = CodeAgent(
    tools=[rag_tool], 
    model=model,
    planning_interval=3)

# 只传 query
resp = rag_agent.run("Hvornår træder bekendtgørelsen i kraft?")

print(resp)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Hvornår træder bekendtgørelsen i kraft?                                                                         │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
## 1. Facts survey

### 1.1. Facts given in the task
- The task asks for the date when a specific regulation (bekendtgørelse) comes into force.

### 1.2. Facts to look up
- The specific name or identifier of the regulation (bekendtgørelse) in question.
  - Source: The task does not provide this information, so it would need to be provided or clarified.
- The official publication date of the regulation.
  - Source: The Danish Legal Database (retsinformation.dk) or the Danish Ministry of Justice.
- The specific clause or section of the regulation that states the date of entry into force.
  - Source: The Danish Legal Database (retsinformation.dk) or the official gazette (Lovtidend) where the regulation
was published.

### 1.3. Facts to derive
- The exact date when the regulation comes into force based on the information found in the official publication.
  - Reasoning: This requires extracting the relevant information from the regulation text and interpreting it 
correctly.

## 2. Plan
1. Identify the specific name or identifier of the regulation (bekendtgørelse) in question.
2. Look up the official publication date of the regulation on the Danish Legal Database (retsinformation.dk) or the
Danish Ministry of Justice.
3. Retrieve the full text of the regulation from the Danish Legal Database (retsinformation.dk) or the official 
gazette (Lovtidend).
4. Locate the clause or section of the regulation that specifies the date of entry into force.
5. Extract and interpret the exact date when the regulation comes into force.
6. final_answer(answer)


```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  regulation_name = "Lov om sundhedsvæsenets digitale infrastruktur"                                               
  result = rag_tool(query=f"Hvornår træder {regulation_name} i kraft?")                                            
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
{'answer': '[Lov 2025_956  2. stk.] Digitaliseringsstyrelsen forlnger samme dag, som bekendtgrelsen trder i kraft, 
jf.  5, den eksisterede tilladelse til administration af internetdomnet . dk. [Lov 2025_681  73. stk.Stk. 4.] 
Teleklagenvnet og Digitaliseringsstyrelsen kan stille krav om, hvordan og i hvilken form oplysninger og materiale 
skal afgives. [Lov 2025_613  15. stk.] Digitaliseringsstyrelsen fastlgger, hvordan ansgning om udbetaling af 
tilskud skal indgives.', 'citations': [{'law_id': '2025_956', 'chapter': '', 'paragraph': '§ 2.', 'section': '', 
'char_offset': 0, 'distance': 0.42153823375701904}, {'law_id': '2025_681', 'chapter': 'Kapitel 28', 'paragraph': '§
73.', 'section': 'Stk. 4.', 'char_offset': 0, 'distance': 0.3979194164276123}, {'law_id': '2025_613', 'chapter': 
'Kapitel 6', 'paragraph': '§ 15.', 'section': '', 'char_offset': 0, 'distance': 0.4568995237350464}]}

Out: None

...

[Step 8: Duration 12.46 seconds| Input tokens: 38,137 | Output tokens: 1,057]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 9 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
402 Client Error: Payment Required for url: https://router.huggingface.co/together/v1/chat/completions (Request ID:
Root=1-68699fd6-33cb483c4f2c80322688aafc;bd485919-d928-4609-ba6c-94d0cd8dac14)

You have exceeded your monthly included credits for Inference Providers. Subscribe to PRO to get 20x more monthly 
included credits.

[Step 9: Duration 0.44 seconds]

AgentGenerationError: Error in generating model output:
402 Client Error: Payment Required for url: https://router.huggingface.co/together/v1/chat/completions (Request ID: Root=1-68699fd6-33cb483c4f2c80322688aafc;bd485919-d928-4609-ba6c-94d0cd8dac14)

You have exceeded your monthly included credits for Inference Providers. Subscribe to PRO to get 20x more monthly included credits.